# 04 DeepSeek V3 架构走读：MoE + MLA

## 简介

本笔记本介绍 DeepSeek V3 的两大核心创新：
- **MLA (Multi-head Latent Attention)**: 低秩压缩 KV Cache，大幅降低推理显存
- **MoE (Mixture of Experts)**: 稀疏激活，以较低的推理计算量获得海量参数

我们将逐步走读 MLA 的解耦 RoPE 设计和 MoE 的 Top-K 路由机制。

## 1. 导入

In [ ]:
import torch
from models.deepseek_v3.config import DeepSeekV3Config
from models.deepseek_v3.mla import MultiHeadLatentAttention
from models.deepseek_v3.moe import Router, SharedExpert, RoutedExpert, MoELayer
from models.deepseek_v3.model import DeepSeekV3Block, DeepSeekV3Model, DeepSeekV3ForCausalLM
from models.common.attention import MultiHeadAttention, GroupedQueryAttention

# 小型演示配置
config = DeepSeekV3Config(
    dim=128, n_heads=4, n_layers=2,
    kv_lora_rank=64, qk_rope_head_dim=16,
    n_routed_experts=8, n_shared_experts=1, n_activated_experts=2,
    moe_intermediate_dim=256, max_seq_len=64,
)
print(f"DeepSeek V3 配置:")
print(f"  dim={config.dim}, heads={config.n_heads}, head_dim={config.head_dim}")
print(f"  kv_lora_rank={config.kv_lora_rank}, qk_rope_head_dim={config.qk_rope_head_dim}")
print(f"  experts: {config.n_routed_experts} routed + {config.n_shared_experts} shared")

### 🧠 直觉理解：MLA 低秩压缩

MLA 的核心思想像**文件压缩**：原始 KV Cache 就像未压缩的 BMP 图片，每个像素都完整存储；MLA 则像 JPEG 压缩——用一个低维的"潜在表示"来近似原始数据，推理时只存储这个压缩版，需要时再解压还原。虽然会损失一点精度，但存储空间节省了 80-90%！

**类比**：想象你要记住 32 个人的详细资料（MHA），但只需要记住 1 个压缩编码（MLA）。虽然从编码还原回完整资料会有微小误差，但大多数情况下足够用了，而记忆负担大幅减轻。

### 📐 数学原理：MLA 低秩压缩

MLA 将 KV 投影分解为两步：

**压缩**（推理时存储）：
$$c_{KV} = W_{down}^{KV} \cdot x \in \mathbb{R}^{d_{lora}}$$

**上投影**（计算时还原）：
$$K = W_{up}^{K} \cdot c_{KV} \in \mathbb{R}^{n_h \times d_h}$$
$$V = W_{up}^{V} \cdot c_{KV} \in \mathbb{R}^{n_h \times d_h}$$

**KV Cache 对比**：
- MHA: 存储 $2 \times n_h \times d_h$ 个元素/每token
- MLA: 存储 $d_{lora} + d_{rope}$ 个元素/每token

当 $d_{lora} \ll n_h \times d_h$ 时，压缩率极高。

## 2. MLA (Multi-head Latent Attention) 原理

MLA 的核心思想：
1. **低秩压缩 KV**: 将输入压缩到 kv_lora_rank 维的潜在向量，推理时只存储这个压缩向量
2. **解耦 RoPE**: 将位置编码施加到 K 的一小部分维度，避免位置信息在压缩中丢失

```
标准 MHA KV Cache: 每 token 存 n_heads × head_dim (K) + n_heads × head_dim (V)
MLA KV Cache:     每 token 存 kv_lora_rank (压缩潜在向量) + qk_rope_head_dim (解耦 RoPE)
```

In [ ]:
mla = MultiHeadLatentAttention(config)
x = torch.randn(2, 8, config.dim)

print("=" * 60)
print("MLA 逐步走读")
print("=" * 60)
print(f"输入 x: {list(x.shape)}")

# 步骤 1: Q 投影
q = mla.w_q(x)
q = q.view(2, 8, config.n_heads, config.head_dim).transpose(1, 2)
print(f"\n1. Q 投影后分头: {list(q.shape)}")

# 步骤 2: KV 压缩 (关键!)
kv_a = mla.w_kv_a(x)
kv_latent = kv_a[:, :, :config.kv_lora_rank]
k_rope_raw = kv_a[:, :, config.kv_lora_rank:]
print(f"2. KV 压缩输出: {list(kv_a.shape)}")
print(f"   - 潜在向量 (存储到 Cache): {list(kv_latent.shape)}")
print(f"   - RoPE 部分 (存储到 Cache): {list(k_rope_raw.shape)}")

# 步骤 3: 上投影 K, V
k = mla.w_k_b(kv_latent)
k = k.view(2, 8, config.n_heads, config.head_dim).transpose(1, 2)
v = mla.w_v_b(kv_latent)
v = v.view(2, 8, config.n_heads, config.head_dim).transpose(1, 2)
print(f"\n3. K 上投影: {list(k.shape)}")
print(f"4. V 上投影: {list(v.shape)}")

# 步骤 4: 解耦 RoPE
q_nope = q[:, :, :, :-config.qk_rope_head_dim]
q_rope = q[:, :, :, -config.qk_rope_head_dim:]
print(f"\n5. Q 非 RoPE 部分: {list(q_nope.shape)}")
print(f"6. Q RoPE 部分:    {list(q_rope.shape)}")

q_rope = mla._apply_rope(q_rope)
k_rope = mla._apply_rope(k_rope_raw)
k_rope = k_rope.unsqueeze(1).expand(2, config.n_heads, 8, config.qk_rope_head_dim)
print(f"7. K RoPE (广播到所有头): {list(k_rope.shape)}")

# 步骤 5: 最终输出
out = mla(x, use_causal_mask=True)
print(f"\n最终输出: {list(out.shape)} (与输入一致: {out.shape == x.shape})")

## 3. KV Cache 大小对比: MHA vs GQA vs MLA

In [ ]:
def kv_cache_per_token_bytes(n_heads, n_kv_heads, head_dim, kv_lora_rank, rope_dim, dtype=2):
    """计算每 token 的 KV Cache 字节数 (float16)
    
    MHA/GQA: 存完整 K + V  (各 n_kv_heads * head_dim 维)
    MLA:     存潜在向量 kv_lora_rank + RoPE 键 rope_dim
    """
    mha_bytes = 2 * n_heads * head_dim * dtype
    gqa_bytes = 2 * n_kv_heads * head_dim * dtype
    mla_bytes = (kv_lora_rank + rope_dim) * dtype
    return mha_bytes, gqa_bytes, mla_bytes

head_dim = 128
n_heads = 32
n_kv_heads = 8
kv_lora_rank = 512
rope_dim = 64

mha_b, gqa_b, mla_b = kv_cache_per_token_bytes(
    n_heads, n_kv_heads, head_dim, kv_lora_rank, rope_dim
)
print(f"每 token KV Cache 大小 (float16):")
print(f"  MHA ({n_heads} heads): {mha_b} bytes = {mha_b/1024:.2f} KB")
print(f"  GQA ({n_kv_heads} KV heads): {gqa_b} bytes = {gqa_b/1024:.2f} KB")
print(f"  MLA ({kv_lora_rank}+{rope_dim}):  {mla_b} bytes = {mla_b/1024:.2f} KB")
print(f"\n  MLA 相对 MHA 节省: {(1 - mla_b/mha_b)*100:.1f}%")
print(f"  MLA 相对 GQA 节省: {(1 - mla_b/gqa_b)*100:.1f}%")

### 📊 MLA 压缩率对比图

可视化 MLA 相对于 MHA 和 GQA 的 KV Cache 压缩效果。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# DeepSeek-V3 参数
head_dim = 128
n_heads = 32
n_kv_heads = 8
kv_lora_rank = 512
rope_dim = 64
dtype = 2  # fp16

# 每 token KV Cache bytes
mha_bytes = 2 * n_heads * head_dim * dtype
gqa_bytes = 2 * n_kv_heads * head_dim * dtype
mla_bytes = (kv_lora_rank + rope_dim) * dtype

labels = ['MHA', 'GQA', 'MLA']
sizes_kb = [mha_bytes/1024, gqa_bytes/1024, mla_bytes/1024]
colors = ['#e74c3c', '#f39c12', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 绝对大小
bars = axes[0].bar(labels, sizes_kb, color=colors, width=0.5, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, sizes_kb):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                 f'{val:.1f} KB', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_ylabel('每 token KV Cache (KB)', fontsize=12)
axes[0].set_title('KV Cache 绝对大小对比', fontsize=14)
axes[0].set_ylim(0, max(sizes_kb) * 1.2)
axes[0].grid(axis='y', alpha=0.3)

# 右图: 相对 MHA 的比例
ratios = [s / sizes_kb[0] * 100 for s in sizes_kb]
bars2 = axes[1].bar(labels, ratios, color=colors, width=0.5, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars2, ratios):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_ylabel('相对 MHA 的比例 (%)', fontsize=12)
axes[1].set_title('KV Cache 压缩率对比', fontsize=14)
axes[1].set_ylim(0, 120)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 🧠 直觉理解：MoE 稀疏激活

MoE 就像一个大型医院：虽然医院有 256 个科室（专家），但每个病人（token）只需要看 2-3 个科室。医院不需要每个病人都走遍所有科室，而是通过分诊台（Router）把病人引导到最相关的几个科室。 这样，医院的总"知识容量"（参数量）可以非常大，但每个病人的"就诊时间"（计算量）保持可控。

**关键洞察**：MoE 打破了"参数量 = 计算量"的传统绑定。DeepSeek-V3 有 671B 总参数，但每个 token 只激活约 37B 参数——用 5.5% 的计算量获得了 671B 的知识容量。

### 📐 数学原理：MoE Top-K 路由

MoE 层的输出为：

$$\text{MoE}(x) = \text{SharedExpert}(x) + \sum_{i \in \text{TopK}} s_i \cdot \text{Expert}_i(x)$$

其中路由得分通过 softmax 计算：

$$s_i = \frac{\exp(g_i / T)}{\sum_{j \in \text{TopK}} \exp(g_j / T)}, \quad g_i = (W_{gate} \cdot x)_i$$

**负载均衡辅助损失**：

$$L_{aux} = \alpha \cdot \sum_{i=1}^{N} f_i \cdot p_i$$

其中 $f_i$ 是专家 $i$ 被选中的频率，$p_i$ 是平均路由概率。这防止某些专家被过度使用。

## 4. MoE (Mixture of Experts) 原理

MoE 的核心组件:
1. **Router (Top-K 门控)**: 为每个 token 选择得分最高的 K 个专家
2. **Shared Expert**: 所有 token 都通过的共享专家
3. **Routed Experts**: 只有被路由到的 token 才经过的专家

```
输出 = SharedExpert(x) + Σ_{i in TopK} score_i × RoutedExpert_i(x)
```

In [ ]:
# 演示 Router
router = Router(dim=config.dim, n_experts=config.n_routed_experts,
                top_k=config.n_activated_experts)
x = torch.randn(2, 4, config.dim)

scores, indices = router(x)
print(f"Router 输入: {list(x.shape)}")
print(f"Router 得分: {list(scores.shape)} (Top-{config.n_activated_experts} 归一化得分)")
print(f"Router 索引: {list(indices.shape)} (选中的专家编号)")
print(f"\n第一个 token 的路由结果:")
print(f"  得分: {scores[0, 0].detach().numpy().round(3)}")
print(f"  专家: {indices[0, 0].tolist()}")

In [ ]:
# 完整 MoE Layer 前向传播
moe_layer = MoELayer(config)
x = torch.randn(2, 8, config.dim)

print("=" * 60)
print("MoE Layer 前向传播")
print("=" * 60)
out = moe_layer(x)
print(f"输入: {list(x.shape)}")
print(f"输出: {list(out.shape)}")

# 统计专家选择分布
_, indices = moe_layer.router(x)
flat_indices = indices.flatten()

print(f"\n专家选择分布 (共 {flat_indices.numel()} 个 token, 每 token 选 {config.n_activated_experts} 个):")
for e in range(config.n_routed_experts):
    count = (flat_indices == e).sum().item()
    bar = "█" * (count // 2) if count > 0 else ""
    print(f"  Expert {e}: {count:3d} tokens {bar}")

print(f"\n共享专家: 所有 {x.shape[0] * x.shape[1]} 个 token 都经过")

### 📊 MoE 路由分布可视化

用热力图展示 token 被路由到各 expert 的分布情况。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 模拟 MoE 路由分布
np.random.seed(42)
n_tokens = 64
n_experts = 8
top_k = 2

# 生成路由得分和选择
router_logits = np.random.randn(n_tokens, n_experts)
router_probs = np.exp(router_logits) / np.exp(router_logits).sum(axis=1, keepdims=True)

# Top-K 选择
routing_matrix = np.zeros((n_tokens, n_experts))
top_k_indices = np.argsort(router_logits, axis=1)[:, -top_k:]
for i in range(n_tokens):
    for j in top_k_indices[i]:
        routing_matrix[i, j] = router_probs[i, j]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图: Token-Expert 路由热力图
im = axes[0].imshow(routing_matrix, cmap='YlOrRd', aspect='auto')
axes[0].set_xlabel('Expert 编号', fontsize=12)
axes[0].set_ylabel('Token 编号', fontsize=12)
axes[0].set_title('Token-Expert 路由热力图', fontsize=14)
plt.colorbar(im, ax=axes[0], label='路由概率')

# 右图: 每个 Expert 被选中的次数
expert_counts = (routing_matrix > 0).sum(axis=0)
ideal = n_tokens * top_k / n_experts
colors = ['#e74c3c' if c > ideal * 1.5 else '#2ecc71' if c < ideal * 0.5 else '#3498db'
          for c in expert_counts]
bars = axes[1].bar(range(n_experts), expert_counts, color=colors, edgecolor='black', linewidth=0.5)
axes[1].axhline(y=ideal, color='red', linestyle='--', linewidth=2, label=f'理想均匀分布={ideal:.0f}')
axes[1].set_xlabel('Expert 编号', fontsize=12)
axes[1].set_ylabel('被选中的 Token 数', fontsize=12)
axes[1].set_title('Expert 负载分布', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. DeepSeek V3 Full Forward Pass

In [ ]:
# 完整模型
model = DeepSeekV3Model(config)
lm = DeepSeekV3ForCausalLM(config)

tokens = torch.randint(0, config.vocab_size, (2, 16))

h = model(tokens, use_causal_mask=True)
logits = lm(tokens, use_causal_mask=True)

print(f"DeepSeekV3Model:  {list(tokens.shape)} -> {list(h.shape)}")
print(f"DeepSeekV3ForCausalLM: {list(tokens.shape)} -> {list(logits.shape)}")

print(f"\n总参数量: {sum(p.numel() for p in lm.parameters()):,}")
print(f"  但每个 token 只激活 1 共享专家 + {config.n_activated_experts}/{config.n_routed_experts} 路由专家")
print(f"  实际激活参数: 仅约 1/({config.n_routed_experts}) 的总参数")

## 6. MoE 激活参数分析

In [ ]:
# 计算一个路由专家的参数量
expert = moe_layer.routed_experts[0]
expert_params = sum(p.numel() for p in expert.parameters())
shared_params = sum(p.numel() for p in moe_layer.shared_expert.parameters())

total_expert_params = shared_params + config.n_routed_experts * expert_params
active_expert_params = shared_params + config.n_activated_experts * expert_params

print(f"MoE 专家参数量分析:")
print(f"  单个路由专家参数: {expert_params:,}")
print(f"  共享专家参数: {shared_params:,}")
print(f"  所有专家总参数: {total_expert_params:,}")
print(f"  每个 token 激活参数: {active_expert_params:,}")
print(f"  激活率: {active_expert_params/total_expert_params*100:.1f}% "
      f"(激活 {config.n_activated_experts + 1}/{config.n_routed_experts + 1} 个专家)")

print(f"\n★ MoE 的核心优势: 参数总量大 (知识容量大) 但单次激活量小 (推理快)")

## 7. 关键要点总结

1. **MLA 低秩压缩**: 将 KV Cache 从 `n_heads × head_dim` 压缩到 `kv_lora_rank`，节省 80-90% KV Cache 内存
2. **解耦 RoPE**: 只对 K 的一小部分维度施加旋转位置编码，避免压缩破坏位置信息
3. **MoE 稀疏激活**: 每个 token 只激活 Top-K 个专家，参数总量大但推理计算量小
4. **路由器负载均衡**: 实际训练中需添加辅助损失防止某些专家被过度使用
5. **MLA + MoE 组合**: MLA 降低注意力内存，MoE 降低 FFN 计算量，两者结合实现高效大模型

## 📝 练习题

### 思考题
1. MLA 的解耦 RoPE 设计为什么要将位置编码施加到 K 的一小部分维度，而不是全部维度？如果全部维度都施加 RoPE 会怎样？

### 编程题
2. 实现一个简化版 MLA：给定输入 x(dim=64)，压缩到 d_lora=16 维，再上投影回 K/V。对比完整 KV Cache 和压缩后的内存占用。

### 分析题
3. DeepSeek-V3 有 256 个路由专家 + 1 个共享专家，每个 token 激活 Top-8。计算：(a) 总参数量与激活参数量的比值；(b) 如果改为 Top-4，推理速度提升多少？质量可能下降多少？